###  Installation de l'environnement

In [7]:
%pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] Aucun fichier ou dossier de ce nom: 'requirements.txt'
Note: you may need to restart the kernel to use updated packages.


### Connexion à la DB DuckDB

In [8]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Connexion à la DB / Import des Data


In [9]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

In [10]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 4 tables in the database:

1. all_events
2. loaded_files
3. user_events
4. user_segments_kmeans


In [18]:
def view_table(table_name: str):
    # Examine the all_events table
    table_data = con.sql(f"""
        SELECT * FROM {table_name}
    """)
    table_data.show()

    # Show count of records in all_events
    record_count = con.sql(f"""
        SELECT COUNT(*) as total_count FROM {table_name}
    """)
    record_count.show()

    # Examine the loaded_files table
    print("\nContents of loaded_files table:")
    loaded_files_data = con.sql("""
        SELECT * FROM loaded_files
    """)
    loaded_files_data.show()


In [12]:
def drop_table(table_name: str) -> bool:
    
    # Drop table if it exists
    con.sql(f"DROP TABLE IF EXISTS {table_name}")

    print(f"Table {table_name} dropped")


In [19]:
view_table("all_events", limit=5000)

TypeError: view_table() got an unexpected keyword argument 'limit'

In [20]:
view_table("user_events")

┌───────────┬──────────────┬─────────────┬─────────────────┬─────────────────────────┬─────────────┬────────────────────┬─────────────────────┬──────────────────────┬───────────────────────┬───────────────────────┐
│  user_id  │ total_events │ total_views │ total_purchases │ avg_time_between_events │ total_spent │     avg_basket     │   last_event_time   │   conversion_rate    │    purchase_ratio     │ days_since_last_event │
│  varchar  │    int64     │   double    │     double      │         double          │   double    │       double       │      timestamp      │        double        │        double         │         int64         │
├───────────┼──────────────┼─────────────┼─────────────────┼─────────────────────────┼─────────────┼────────────────────┼─────────────────────┼──────────────────────┼───────────────────────┼───────────────────────┤
│ 575829553 │           36 │        35.0 │             0.0 │      230035.65714285715 │         0.0 │                0.0 │ 2020-02-25 12:55:3

In [15]:
view_table("user_segments", limit=5000)

First 5000 rows of user_segments table:


CatalogException: Catalog Error: Table with name user_segments does not exist!
Did you mean "user_segments_kmeans"?

In [ ]:
drop_table("user_events")

Table user_events dropped
